In [ ]:
import os
import numpy as np
from multiprocessing import get_context, current_process
from cellpose import models
import torch
import logging

### Runs the first DAPI image (best quality) over the last image (worst quality) so there aren't any ghost cells in the segmentation, this
### causes loss of cells in some images. The segmentation is done using Cellpose

def init_cellpose(gpu_id):
    """
    Initialize a Cellpose model on the desired GPU (or CPU if gpu_id is None/negative).
    Must be called inside each worker process.
    """
    global model

    if gpu_id is None or gpu_id < 0:
        device = torch.device("cpu")
        use_gpu = False
    else:
        # Set the current CUDA device for this process and create a torch.device object
        torch.cuda.set_device(gpu_id)
        device = torch.device(f"cuda:{gpu_id}")
        use_gpu = True

    # Pass a torch.device object to Cellpose (Cellpose expects a device-like object)
    model = models.Cellpose(model_type='nuclei', gpu=use_gpu, device=device)

def worker(file_list, gpu_id, save_dir):
    """
    This is the actual script, the rest just revolves around running this piece properly.
    Here we are running cellpose then saving them to the desired directory.
    """
    logging.info(f"Processing begun")
    init_cellpose(gpu_id)
    for file_path in file_list:
        logging.info(f"Processing file: {file_path}")
        arr = np.load(file_path)
        DAPI = arr[0, 0, :, :]
        intensity_images = arr[:, :, :, :]

        masks, flows, styles, diams = model.eval(
            DAPI, diameter=20, channels=[0,0],
            cellprob_threshold=0.0, flow_threshold=0.4, niter=2000
        )

        base = os.path.basename(file_path).replace(".npy", "")
        np.save(os.path.join(save_dir, f"{base}_masks.npy"), masks)
        np.save(os.path.join(save_dir, f"{base}_intensity.npy"), intensity_images)
        logging.info(f"[{current_process().name} GPU {gpu_id}] Processed: {file_path}")

def run_workers(directory=None, save_dir=None, n_gpus=1, n_workers=None, max_files=None, file_list=None):
    """
    Parallelize Cellpose on a set of files.
    If `file_list` is provided, it uses that list instead of reading from `directory`.
    """
    if file_list is None:
        if directory is None:
            raise ValueError("Either 'directory' or 'file_list' must be provided.")
        files = sorted([
            os.path.join(directory, f)
            for f in os.listdir(directory)
            if f.endswith(".npy")
        ])
    else:
        files = file_list

    processed_bases = set()
    for f in os.listdir(save_dir):
        if f.endswith("_masks.npy"):
            base = f.replace("_masks.npy", "")  # full base name
            intensity_file = os.path.join(save_dir, f"{base}_intensity.npy")
            if os.path.exists(intensity_file):
                processed_bases.add(base)  # use full base, not just TMA#

    files_to_run = [
        f for f in files
        if os.path.basename(f).replace("_aligned.npy", "") not in processed_bases
    ]
    if max_files:
        files_to_run = files_to_run[:max_files] ###Here you can edit the indexing of the files to make sure you don't rerun everything
    os.makedirs(save_dir, exist_ok=True)
    if len(files_to_run) == 0:
        return []
    if n_workers is None:
        n_workers = min(len(files_to_run), n_gpus)

    # chunk files across workers via striding
    chunks = [files_to_run[i::n_workers] for i in range(n_workers)]

    ctx = get_context('spawn')
    procs = []
    for i, chunk in enumerate(chunks):
        if not chunk:
            continue
        gpu_id = i % n_gpus
        p = ctx.Process(target=worker, args=(chunk, gpu_id, save_dir), name=f"worker-{i}")
        p.start()
        procs.append(p)

    for p in procs:
        p.join()

    return files_to_run
